In [ ]:
import sys
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, LogLocator, NullFormatter
from glob import glob
from sklearn.utils import resample

In [ ]:
import pymbar
sys.path.append("../")
from pymbar.mbar_pmf import mbar_pmf

In [ ]:
pwd

In [ ]:
n_windows = 40
val_min = -1.975
val_max = 1.925

In [ ]:
val0_k = np.linspace(val_min, val_max, n_windows)
print(val0_k)

In [ ]:
plt.figure(figsize=(15,2), dpi=100)
for i in range(n_windows):
    plt.axvline(val0_k[i], alpha=0.2)
    fnames = sorted(glob('../%02d/step6.02equilibration.cv' % i))
    print(fnames)
    arrays = [np.loadtxt(f, usecols=1)[::] for f in fnames[:]]
    val_kn = np.concatenate(arrays)[:] 
    plt.hist(val_kn, alpha=0.4) 
plt.xlabel("R1 - R2 (Å)", fontsize=14)
plt.ylabel("Count", fontsize=14)
plt.savefig("cv_histogramstep6.png")

In [ ]:
val_kn = []
for i in range(n_windows):
    fnames = sorted(glob('../%02d/step6.02equilibration.cv' % i))
    arrays = [np.loadtxt(f, usecols=1)for f in fnames[:]]
    val_kn.append(np.concatenate(arrays)[::])
val0_k = np.linspace(val_min, val_max, n_windows)
K_k = np.ones(n_windows) * 300.0
nbins = n_windows -1 

In [ ]:
for i in range(n_windows):
    print(i)
    print("Window %02d:" % i, pymbar.timeseries.subsampleCorrelatedData(val_kn[i], conservative=True))

In [ ]:
mbar = mbar_pmf(val_kn, val0_k, K_k, 300.0)

In [ ]:
step = "CHO_nod3"
bin_centers, f_i, df_i, reweighting_entropy = mbar.get_pmf(val_min, val_max, nbins)
bin_centers, f_i, df_i, reweighting_entropy = mbar.get_pmf(val_min, val_max, nbins, uncertainties='from-specified', pmf_reference=f_i[:20].argmin())
np.savetxt(f"freefile_mbar{step}", np.column_stack((bin_centers, f_i, df_i)))
np.savetxt(f"reweighting_entropy{step}", reweighting_entropy)


In [ ]:
freefile_mbar = [
                'freefile_mbarCHO_nod3'
                ]

for i in range(len(freefile_mbar)):
    initial = np.loadtxt(freefile_mbar[i])
    # legend = labels[i] +
    legend = '$\Delta$A$^\ddag$ = ' + str(np.round(initial[:,1].max() - initial[:10,1].min(),1)) + ' $\pm$ ' + str(np.round(initial[initial[:,1].argmax()][2],1)) + ' kcal/mol'
    plt.errorbar(initial[:,0], initial[:,1] - initial[:10,1].min(), yerr=initial[:,2], linewidth=1, label=legend)
    plt.legend(fontsize=12)
    print(round(initial[:,1].max() - initial[:10,1].min(),1), round(initial[initial[:,1].argmax()][2],1))
plt.xlabel("R1 - R2 (Å)", fontsize=14)
plt.ylabel("Potential of Mean Force (kcal/mol)", fontsize=14)
plt.grid (linestyle='--', alpha=0.4)            
plt.savefig("pmfstep6.png", bbox_inches='tight', dpi=300)
plt.show()

